# Lab 5: Tidy Data and Reshaping

**DSA 405 · Week 5**

| | |
|---|---|
| **In class** | Friday, Sep 18 |
| **A5 due** | Thursday, Sep 24, 11:59 PM |
| **File** | `nc_schools_dirty.xlsx` |
| **Also this week** | **Bench Check 1** window opens (slots this week through Week 7) |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

This week is built on Wickham's three rules for tidy data: every variable is a column,
every observation is a row, and every type of observational unit gets its own table.
They sound almost too simple to be useful, but the same three rules apply in R, SQL,
and Polars, and they explain something you have probably already noticed: a spreadsheet
formatted to look nice for human readers is hard for a computer to use.

The file we're working with is a school test-score workbook that violates many
standards: headers on row 4, three sheets, grade columns laid out sideways, footnote
markers inside the numbers, and one row that is not a school.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
# Part 1: Explore (in class)

## Task 1.1: Load it right, then look at the layout

We've seen this file's header problem before, in Week 2. Load it with the correct
header row, and this time look at the *shape* of the table rather than the numbers:

In [ ]:
schools = load("nc_schools_dirty.xlsx", "excel", header=3)

print(schools.shape)
print(list(schools.columns))
schools.head(3)

The layout violates the first rule. `grade_3_reading`, `grade_4_reading`,
`grade_5_reading`, and the rest of the measure columns each encode **two variables**,
grade level and subject, in the header row. A variable (grade) is spread across
columns, so an easy-sounding question like "average reading score by grade" makes you
need to read three columns instead of grouping just one.

Now check the sheets:

In [ ]:
all_sheets = load("nc_schools_dirty.xlsx", "excel", header=3, sheet_name=None)
for name, df in all_sheets.items():
    print(f"{name}: {df.shape}, columns: {list(df.columns)[:4]} ...")

The second sheet contains the same data for a second year under **different column
names** (`LEA`, `school_name`, `g3_read`), so two sheets that should be stacked into
one table cannot be combined until the column names are made to match. Expect to see this same problem in your own project data; it is very common when data comes from more than one system.
Today we work with the
2023–24 sheet.

## Task 1.2: Melt, wide to long

`melt` reshapes a table from wide to long. Here it turns the six grade/subject columns
into two: a *measure* name and a *value*.

In [ ]:
MEASURES = ["grade_3_reading", "grade_4_reading", "grade_5_reading",
            "grade_3_math", "grade_4_math", "grade_5_math"]

long = schools.melt(id_vars=["district", "school", "enrollment"],
                    value_vars=MEASURES,
                    var_name="measure", value_name="pct_proficient")

print(f"{len(schools)} rows x {len(MEASURES)} measures = {len(long)} rows")
assert len(long) == len(schools) * len(MEASURES)
long.head()

Before trusting the reshape, verify the arithmetic: 97 × 6 = 582, and the assert
checks it. (An **assert** is a line of code that states something that must be true;
Python stops with an error message if it is not.) Reshaping only moves data around; it never adds or removes values. So the
row counts must multiply out exactly. If they do not, some rows were lost or duplicated
during the reshape, and you should find out which before going any further.

Questions that used to require reading three columns can now be answered with one
`groupby`, the pandas method that computes a summary (like a mean or a count) for
each group of rows:

In [ ]:
long[["grade", "subject"]] = long.measure.str.extract(r"grade_(\d)_(\w+)")
scores = pd.to_numeric(long.pct_proficient, errors="coerce")

print(long.assign(v=scores).groupby("grade").v.mean().round(1))

(The **regex**, a pattern that describes what a piece of text looks like, is the same
pattern from last week. `errors="coerce"` converted some cells to `NaN`, the value
pandas uses to mean "missing"; those cells are the subject of A5.)

---
## Checkpoint: submit before leaving class

1. Which of the 3 tidy rules does the wide layout violate? What specific question is
   difficult to answer because of the non-tidy layout?
2. What is the melt arithmetic for this sheet (rows before × number of measures = rows
   after), and did the assert pass?
3. One row of this sheet does not look like an observation. Name that row.

*Answers here.*

---
# Part 2: A5 (Tidy & Reshape)

Four tasks. They are the take-home half of the week.

## Task 2.1: The row that is not an observation

Rule two says every row is an observation. One row of this sheet is an aggregate (a
total computed from the other rows), not a school. If it stays in the table, every
statistic you compute from the table will be wrong.

1. Find it. The profiling methods from Week 2 will find it quickly, and so will sorting
   the table.
2. Show what the extra row does to a statistic: report the **maximum grade-3 reading
   score** twice, once with the row included and once with it removed. One of the two
   numbers is larger than 100, so it cannot be a percentage. That is strong evidence
   you found the right row.
3. Remove the row, report how many actual schools remain, and record the removal in the
   cleaning log with a count and a reason.

In [ ]:
# your search

## Task 2.2: Four markers, four meanings

The score cells contain more than numbers. Collect every **non-numeric marker** in the
six measure columns; a marker here is a symbol or short code written in a cell instead
of a number. The `Notes` sheet explains what each marker means. There are four:

For each marker, report how many cells contain it, what it means, and what value it
should become in a numeric column. Then a question to think carefully about: **one of
the four markers is a different kind of marker from the other three.** Which one, and
why? (Consider whether a real value exists for the cell and the marker only hides it.) The distinction matters
because the correct numeric replacement depends on it.

In [ ]:
# your marker counts

*Answers here.*

## Task 2.3: The question tidy makes easy

Using the tidy long table (TOTAL row removed, markers handled, values numeric): compute
the mean grade-3 reading proficiency **by district**, sorted. Report the highest and
lowest districts with their values.

Then reshape once more: use `pivot_table` to turn the district means back to wide,
districts as rows, measures as columns, and show the result. The two layouts have
different uses: the long table is the one you compute on, and the wide table is the one
a person reads.

In [ ]:
# your groupby and pivot

## Task 2.4: Who disappears when suppressed cells are dropped

The `<5` marker means *fewer than five students tested; value suppressed for privacy*
(the real value was removed so that no student can be identified). A common shortcut is
to drop every school that has any suppressed cell. This task measures what that
shortcut removes from the data.

1. Split the schools into two groups: schools with at least one suppressed cell, and
   schools with none. Report the **mean enrollment** of each group.
2. Compute mean proficiency across all reported cells, and again using only the
   never-suppressed schools. Report both.
3. Then write a paragraph: which schools disappear from the data (their size and
   location), which direction the average moves, and why the schools that disappear
   matter for a policy question about school performance. Propose one concrete
   reporting practice that keeps the suppressed schools visible. Concrete means someone
   else could follow it next month without asking you what you meant.

In [ ]:
# your split and comparison

*Paragraph here.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A5_[yourUnityID].ipynb`
4. Upload to the **A5** space on Moodle.

The **Checkpoint** section goes separately to **Week 5 In-Class Activity** before the
end of class on Friday. A5 is due **Thursday, Sep 24, 11:59 PM**.